# AI Slop Prompt Rewriter (Colab, GPU, open-source)

This notebook builds a **first-pass prompt rewriter** for **text-generation prompts**.

Pipeline:
1. Use a **small open-source generator** (`distilgpt2`) to simulate outputs quickly.
2. Score generated outputs for **AI-likeness** using:
   - **Pangram/EditLens** if your Hugging Face access is approved, or
   - a **public fallback detector** if Pangram is still gated.
3. Create synthetic `(original_prompt -> better_rewritten_prompt)` training pairs by ranking prompt variants on:
   - lower AI-likeness score,
   - better meaning preservation,
   - slightly better style.
4. Fine-tune a small rewriter model (`flan-t5-small`).

This version is set up to avoid the gating and package-version issues we hit earlier.


## How to run

Run the notebook **from top to bottom**.

Important:
- The first code cell **installs pinned package versions and restarts the runtime on purpose**.
- After the restart, start again from the next code cell and continue downward.
- If Pangram access is still pending, the notebook will **automatically fall back** to a public detector.


In [1]:
# =========================
# CELL 1: Clean install + forced restart
# Run this cell ONCE.
# =========================

!pip -q uninstall -y transformers tokenizers huggingface_hub accelerate sentence-transformers safetensors
!pip -q install --no-cache-dir \
  "transformers==4.45.2" \
  "tokenizers==0.20.1" \
  "huggingface_hub==0.25.2" \
  "accelerate==0.34.2" \
  "sentence-transformers==3.2.1" \
  "safetensors==0.4.5" \
  "datasets>=2.21.0" \
  "pandas>=2.2.0" \
  "scikit-learn>=1.5.0"

# import os
# print("Packages installed. Restarting runtime now...")
# os.kill(os.getpid(), 9)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 410.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 404.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 408.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 399.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 388.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 413.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.25.2 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.25.2 which is incompatible.


## Imports and configuration

After the runtime restarts, run the cells below.


In [2]:
import os
import gc
import math
import random
import re
from typing import List, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from huggingface_hub import login
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset, load_dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [16]:
# =========================
# Config
# =========================

# Detector
PRIMARY_SCORER = "pangram/editlens_roberta-large"
FALLBACK_SCORER = "openai-community/roberta-base-openai-detector"

# Models
GEN_MODEL_NAME = "distilbert/distilgpt2"
REWRITER_MODEL_NAME = "google/flan-t5-small"
SIM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Real prompt dataset (public, prompt/response format)
PROMPT_DATASET_NAME = "databricks/databricks-dolly-15k"
PROMPT_DATASET_SPLIT = "train"
USE_CONTEXT_IN_PROMPTS = True

# Dataset size / speed knobs
N_BASE_PROMPTS = 224
N_HELDOUT_PROMPTS = 24
GEN_MAX_NEW_TOKENS = 64
N_CANDIDATES_PER_PROMPT = 4
MAX_REWRITE_GEN_TOKENS = 96

# Training knobs
MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 128
TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 8
LEARNING_RATE = 2e-4

# Optional HF token:
# If later approved for Pangram, set HF_TOKEN in Colab secrets or environment.
HF_TOKEN = os.environ.get("HF_TOKEN", None)


## Load models

This cell:
- tries Pangram first,
- falls back to a public detector if Pangram is gated,
- loads DistilGPT2 for cheap generation,
- loads FLAN-T5-small for rewriting,
- loads MiniLM for semantic similarity.


In [17]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [18]:
if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Logged into Hugging Face.")
    except Exception as e:
        print(f"HF login failed: {e}")
else:
    print("No HF token found. Will try Pangram if access is approved, else fallback.")

def load_seqclf_model(repo_id, token=None):
    tok = AutoTokenizer.from_pretrained(repo_id, token=token, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(repo_id, token=token)
    model.to(device).eval()
    return tok, model

def load_detector_with_fallback(primary_repo, fallback_repo, token=None):
    try:
        print(f"Trying primary scorer: {primary_repo}")
        tok, model = load_seqclf_model(primary_repo, token=token)
        print(f"Loaded primary scorer: {primary_repo}")
        return tok, model, primary_repo, "editlens_regression_like"
    except Exception as e:
        print(f"Primary scorer unavailable: {e}")
        print(f"Falling back to public scorer: {fallback_repo}")
        tok, model = load_seqclf_model(fallback_repo, token=None)
        print(f"Loaded fallback scorer: {fallback_repo}")
        return tok, model, fallback_repo, "binary_classifier"

detector_tokenizer, detector_model, DETECTOR_NAME, DETECTOR_TYPE = load_detector_with_fallback(
    PRIMARY_SCORER,
    FALLBACK_SCORER,
    token=HF_TOKEN
)

print(f"Loading generator: {GEN_MODEL_NAME}")
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME, use_fast=True)
if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_NAME)
gen_model.to(device).eval()

print(f"Loading rewriter: {REWRITER_MODEL_NAME}")
rewriter_tokenizer = AutoTokenizer.from_pretrained(REWRITER_MODEL_NAME, use_fast=True)
rewriter_model = AutoModelForSeq2SeqLM.from_pretrained(REWRITER_MODEL_NAME)
rewriter_model.to(device)

print(f"Loading similarity model: {SIM_MODEL_NAME}")
sim_model = SentenceTransformer(SIM_MODEL_NAME, device=device)

print("\nLoaded successfully:")
print("  Detector:", DETECTOR_NAME, f"({DETECTOR_TYPE})")
print("  Generator:", GEN_MODEL_NAME)
print("  Rewriter:", REWRITER_MODEL_NAME)
print("  Similarity:", SIM_MODEL_NAME)


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
Logged into Hugging Face.
Trying primary scorer: pangram/editlens_roberta-large
Primary scorer unavailable: We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like pangram/editlens_roberta-large is not the path to a directory containing a file named config.json.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
Falling back to public scorer: openai-community/roberta-base-openai-detector


Some weights of the model checkpoint at openai-community/roberta-base-openai-detector were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loaded fallback scorer: openai-community/roberta-base-openai-detector
Loading generator: distilbert/distilgpt2
Loading rewriter: google/flan-t5-small
Loading similarity model: sentence-transformers/all-MiniLM-L6-v2

Loaded successfully:
  Detector: openai-community/roberta-base-openai-detector (binary_classifier)
  Generator: distilbert/distilgpt2
  Rewriter: google/flan-t5-small
  Similarity: sentence-transformers/all-MiniLM-L6-v2


## Scoring and generation helpers

In [5]:
def ai_likeness_score(texts, max_length=512):
    """
    Returns a score in [0,1], where higher = more AI-like.
    Works for Pangram or the fallback detector.
    """
    if isinstance(texts, str):
        texts = [texts]

    enc = detector_tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = detector_model(**enc)
        logits = outputs.logits

    if DETECTOR_TYPE == "editlens_regression_like":
        if logits.shape[-1] == 1:
            scores = torch.sigmoid(logits.squeeze(-1))
        else:
            probs = F.softmax(logits, dim=-1)
            scores = probs[:, -1]
        return scores.detach().cpu().numpy().tolist()

    probs = F.softmax(logits, dim=-1)
    id2label = getattr(detector_model.config, "id2label", None)
    ai_idx = None

    if id2label:
        lowered = {int(k): str(v).lower() for k, v in id2label.items()}
        for idx, label in lowered.items():
            if any(term in label for term in ["ai", "generated", "fake", "machine", "chatgpt"]):
                ai_idx = idx
                break

    if ai_idx is None:
        ai_idx = 1 if probs.shape[-1] > 1 else 0

    scores = probs[:, ai_idx]
    return scores.detach().cpu().numpy().tolist()


def semantic_similarity(texts_a: List[str], texts_b: List[str], batch_size: int = 32) -> List[float]:
    emb_a = sim_model.encode(texts_a, batch_size=batch_size, convert_to_tensor=True, normalize_embeddings=True)
    emb_b = sim_model.encode(texts_b, batch_size=batch_size, convert_to_tensor=True, normalize_embeddings=True)
    sims = (emb_a * emb_b).sum(dim=1)
    return sims.detach().cpu().numpy().tolist()


def lightweight_style_score(texts: List[str]) -> List[float]:
    """
    Small heuristic style score in [0,1].
    Rewards specificity and discourages vague prompt fluff.
    """
    vague_terms = {
        "good", "nice", "great", "interesting", "effective", "various",
        "things", "stuff", "really", "very", "overall", "basically"
    }
    scores = []
    for t in texts:
        words = re.findall(r"\b\w+\b", t.lower())
        if not words:
            scores.append(0.0)
            continue

        length_bonus = min(len(words) / 20.0, 1.0)
        vague_penalty = sum(w in vague_terms for w in words) / max(len(words), 1)
        punctuation_bonus = 0.1 if ":" in t or ";" in t else 0.0
        specificity = max(0.0, min(1.0, 0.65 * length_bonus + punctuation_bonus - 1.5 * vague_penalty))
        scores.append(float(specificity))
    return scores


def generate_from_prompts(prompts, max_new_tokens=80):
    """
    Generate one output per prompt using DistilGPT2.
    """
    if isinstance(prompts, str):
        prompts = [prompts]

    outputs = []

    for p in prompts:
        framed = f"Prompt: {p}\nResponse:"
        toks = gen_tokenizer(
            framed,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding=False,
        )
        toks = {k: v.to(device) for k, v in toks.items()}

        with torch.no_grad():
            out = gen_model.generate(
                **toks,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.9,
                top_p=0.95,
                pad_token_id=gen_tokenizer.eos_token_id,
                eos_token_id=gen_tokenizer.eos_token_id,
            )

        decoded = gen_tokenizer.decode(out[0], skip_special_tokens=True)
        if "Response:" in decoded:
            decoded = decoded.split("Response:", 1)[1].strip()
        outputs.append(decoded)

    return outputs


In [6]:
# Quick smoke test
sample_texts = [
    "Write a heartfelt thank-you note to a professor who helped you through a difficult semester.",
    "Write something good about technology and how it helps society overall."
]
scores = ai_likeness_score(sample_texts)
for t, s in zip(sample_texts, scores):
    print(f"{s:.4f} | {t[:100]}")


0.8964 | Write a heartfelt thank-you note to a professor who helped you through a difficult semester.
0.5177 | Write something good about technology and how it helps society overall.


## Real prompt pool from a dataset

This notebook now uses a real prompt/response dataset instead of a hand-written synthetic prompt list. The primary pool is Dolly 15k because it is public and already organized as instruction / context / response records, which makes it a good fit for prompt rewriting. If you later get access to Pangram's gated dataset, you can still use Pangram for scoring, but this notebook keeps the prompt pool public and runnable.


In [7]:
def normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

def build_prompt_from_row(row: Dict) -> str:
    instruction = normalize_spaces(row.get("instruction", ""))
    context = normalize_spaces(row.get("context", ""))
    if USE_CONTEXT_IN_PROMPTS and context:
        return f"{instruction}\n\nContext:\n{context}"
    return instruction

prompt_ds = load_dataset(PROMPT_DATASET_NAME, split=PROMPT_DATASET_SPLIT)
prompt_df = prompt_ds.to_pandas()[["instruction", "context", "response", "category"]].fillna("")

prompt_df["prompt_text"] = prompt_df.apply(
    lambda r: build_prompt_from_row(r.to_dict()),
    axis=1
)

# Keep prompts that are substantial but still compact enough for fast experimentation
prompt_df = prompt_df[
    prompt_df["prompt_text"].str.len().between(25, 500)
].copy()

prompt_df = prompt_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

base_prompts = prompt_df["prompt_text"].head(N_BASE_PROMPTS).tolist()
heldout_prompts = prompt_df["prompt_text"].iloc[N_BASE_PROMPTS:N_BASE_PROMPTS + N_HELDOUT_PROMPTS].tolist()

print("Loaded prompt dataset:", PROMPT_DATASET_NAME)
print("Rows available after filtering:", len(prompt_df))
print("Categories:", sorted(prompt_df["category"].dropna().unique().tolist())[:8], "...")
print("Training prompts:", len(base_prompts))
print("Held-out prompts:", len(heldout_prompts))
print("\nExample prompt:\n")
print(base_prompts[0])


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Loaded prompt dataset: databricks/databricks-dolly-15k
Rows available after filtering: 10334
Categories: ['brainstorming', 'classification', 'closed_qa', 'creative_writing', 'general_qa', 'information_extraction', 'open_qa', 'summarization'] ...
Training prompts: 224
Held-out prompts: 24

Example prompt:

"Big Bang": what does it mean?


## Candidate prompt rewrites

Instead of appending extra instructions onto the end of the original prompt, this cell now uses FLAN-T5 to rewrite the prompt from scratch. Each candidate is a complete replacement prompt that keeps the original task while aiming for lower AI-like downstream generations, stronger meaning preservation, and better specificity.


In [8]:
def clean_rewrite_candidate(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^(rewrite|rewritten prompt|new prompt)\s*[:\-]\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\d+[\).:-]\s*", "", text).strip()
    return normalize_spaces(text)

def rewrite_instruction_for_model(prompt: str) -> str:
    return (
        "Rewrite the following text-generation prompt from scratch. "
        "Preserve the core meaning and task, but make it more specific, grounded, and natural so it is less likely to produce generic AI-sounding output. "
        "Do not add commentary. Return only the rewritten prompt.\n\n"
        f"Original prompt:\n{prompt}\n\nRewritten prompt:"
    )

def candidate_variants(prompt: str) -> List[str]:
    prompt = normalize_spaces(prompt)
    candidates = []
    temperatures = [0.65, 0.8, 0.95, 1.1]

    # Diverse sampled rewrites
    for temp in temperatures[:N_CANDIDATES_PER_PROMPT]:
        model_input = rewrite_instruction_for_model(prompt)
        toks = rewriter_tokenizer(
            model_input,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        )
        toks = {k: v.to(device) for k, v in toks.items()}

        with torch.no_grad():
            out = rewriter_model.generate(
                **toks,
                max_new_tokens=MAX_REWRITE_GEN_TOKENS,
                do_sample=True,
                temperature=float(temp),
                top_p=0.92,
                num_return_sequences=1,
            )

        cand = rewriter_tokenizer.decode(out[0], skip_special_tokens=True)
        cand = clean_rewrite_candidate(cand)

        if cand and cand.lower() != prompt.lower():
            candidates.append(cand)

    # Add one more deterministic candidate for stability
    if len(candidates) < N_CANDIDATES_PER_PROMPT:
        model_input = rewrite_instruction_for_model(prompt)
        toks = rewriter_tokenizer(
            model_input,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        )
        toks = {k: v.to(device) for k, v in toks.items()}

        with torch.no_grad():
            out = rewriter_model.generate(
                **toks,
                max_new_tokens=MAX_REWRITE_GEN_TOKENS,
                num_beams=4,
                do_sample=False,
                early_stopping=True,
            )

        cand = rewriter_tokenizer.decode(out[0], skip_special_tokens=True)
        cand = clean_rewrite_candidate(cand)
        if cand and cand.lower() != prompt.lower():
            candidates.append(cand)

    # Deduplicate while preserving order
    seen = set()
    out = []
    for c in candidates:
        c = normalize_spaces(c)
        if c and c not in seen:
            seen.add(c)
            out.append(c)

    # Safety fallback
    if not out:
        out = [prompt]

    return out[:N_CANDIDATES_PER_PROMPT]

example_original = base_prompts[0]
print("ORIGINAL:\n", example_original)
print("\nREWRITES:")
for i, cand in enumerate(candidate_variants(example_original), 1):
    print(f"{i}. {cand}")


ORIGINAL:
 "Big Bang": what does it mean?

REWRITES:
1. "Big Bang" is a name for the phrase that the phrase "could be "just a sleeveless sleeveless sleeveless sleeveless" is used by the phrase "could a sleeveless sleeveless sleeveless".
2. "Big Bang" means "i have a great deal on the tangent. It is a real, b-side.


## Build training data

For each real prompt from the dataset, we generate multiple full prompt rewrites, run the downstream generator on each rewritten prompt, score the generated text with the AI-likeness detector, and then keep the rewrite with the best composite objective.


In [9]:
synthetic_rows = []

for prompt in tqdm(base_prompts, desc="Building synthetic dataset"):
    cands = candidate_variants(prompt)
    gen_texts = generate_from_prompts(cands, max_new_tokens=GEN_MAX_NEW_TOKENS)
    ai_scores = ai_likeness_score(gen_texts)
    meaning_scores = semantic_similarity([prompt] * len(cands), cands)
    style_scores = lightweight_style_score(cands)

    final_scores = []
    for ai_s, meaning_s, style_s in zip(ai_scores, meaning_scores, style_scores):
        final_obj = (
            1.00 * (1.0 - ai_s) +   # primary objective: less AI-like output
            0.35 * meaning_s +      # preserve meaning of the original prompt
            0.15 * style_s          # slightly reward clearer / more specific prompts
        )
        final_scores.append(final_obj)

    best_idx = int(np.argmax(final_scores))

    synthetic_rows.append({
        "original_prompt": prompt,
        "rewritten_prompt": cands[best_idx],
        "generated_text": gen_texts[best_idx],
        "ai_score": float(ai_scores[best_idx]),
        "meaning_score": float(meaning_scores[best_idx]),
        "style_score": float(style_scores[best_idx]),
        "final_score": float(final_scores[best_idx]),
    })

df = pd.DataFrame(synthetic_rows)
print(df.head())
print("\nMean ai_score of chosen rewrites:", df["ai_score"].mean())
print("Rows:", len(df))


Building synthetic dataset:   0%|          | 0/224 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Inspect examples

In [ ]:
for i in range(min(8, len(df))):
    row = df.iloc[i]
    print("=" * 100)
    print("ORIGINAL :", row["original_prompt"])
    print("REWRITE  :", row["rewritten_prompt"])
    print(f"AI={row['ai_score']:.4f} | meaning={row['meaning_score']:.4f} | style={row['style_score']:.4f} | final={row['final_score']:.4f}")
    print("GEN TEXT :", row["generated_text"][:300].replace("\n", " "))


## Prepare seq2seq training set

In [ ]:
train_df = df[["original_prompt", "rewritten_prompt"]].copy()

def build_instruction_input(prompt: str) -> str:
    return (
        "Rewrite the following text-generation prompt so that it is more likely to produce less AI-like, "
        "less formulaic output while preserving the original meaning and slightly improving style.\n\n"
        f"Prompt: {prompt}"
    )

train_df["input_text"] = train_df["original_prompt"].apply(build_instruction_input)
train_df["target_text"] = train_df["rewritten_prompt"]

dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]], preserve_index=False)
dataset


In [ ]:
split = dataset.train_test_split(test_size=0.1, seed=SEED)
train_ds = split["train"]
eval_ds = split["test"]

def preprocess_batch(batch):
    model_inputs = rewriter_tokenizer(
        batch["input_text"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
    )
    labels = rewriter_tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess_batch, batched=True, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(preprocess_batch, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=rewriter_tokenizer, model=rewriter_model)


## Fine-tune the rewriter

In [ ]:
output_dir = "./slop_rewriter_ckpt"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=rewriter_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    tokenizer=rewriter_tokenizer,
    data_collator=data_collator,
)

trainer.train()


## Inference helper

In [ ]:
def rewrite_prompt(prompt: str, max_new_tokens: int = 64) -> str:
    model_input = build_instruction_input(prompt)
    toks = rewriter_tokenizer(
        model_input,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LEN
    )
    toks = {k: v.to(device) for k, v in toks.items()}

    with torch.no_grad():
        out = rewriter_model.generate(
            **toks,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False
        )

    return rewriter_tokenizer.decode(out[0], skip_special_tokens=True).strip()


## Evaluate before/after on held-out prompts

Use held-out prompts from the same real prompt dataset instead of a hand-written test list.


In [ ]:
test_prompts = heldout_prompts[:5] if len(heldout_prompts) >= 5 else base_prompts[:5]

rows = []
for p in test_prompts:
    rewritten = rewrite_prompt(p)

    original_gen = generate_from_prompts([p], max_new_tokens=GEN_MAX_NEW_TOKENS)[0]
    rewritten_gen = generate_from_prompts([rewritten], max_new_tokens=GEN_MAX_NEW_TOKENS)[0]

    original_ai = ai_likeness_score([original_gen])[0]
    rewritten_ai = ai_likeness_score([rewritten_gen])[0]
    sim = semantic_similarity([p], [rewritten])[0]

    rows.append({
        "prompt": p,
        "rewritten": rewritten,
        "original_ai_score": float(original_ai),
        "rewritten_ai_score": float(rewritten_ai),
        "delta_lower_is_better": float(rewritten_ai - original_ai),
        "meaning_similarity": float(sim),
    })

results_df = pd.DataFrame(rows)
results_df


In [ ]:
print("Average original AI-likeness :", results_df["original_ai_score"].mean())
print("Average rewritten AI-likeness:", results_df["rewritten_ai_score"].mean())
print("Average delta (negative is good):", results_df["delta_lower_is_better"].mean())
print("Average meaning similarity:", results_df["meaning_similarity"].mean())


## Save the trained rewriter

In [ ]:
save_dir = "/content/slop_prompt_rewriter_model"
rewriter_model.save_pretrained(save_dir)
rewriter_tokenizer.save_pretrained(save_dir)
print("Saved model to:", save_dir)


## Notes

- This version uses a real prompt/response dataset instead of a synthetic starter list.
- Prompt rewrites are now full rewrites generated by FLAN-T5, not just appended style hints.
- Pangram remains the preferred scorer when your Hugging Face access is approved, while the notebook still falls back to a public detector if Pangram access is gated.
